# IMU-Former 预训练 Notebook

基于 EEGFormer 框架迁移到 9 轴 IMU 行为数据的自监督预训练（向量量化 VQ-Transformer）。

**流程**：CSV 数据 → 固定窗数据层 → 分模态 Patching → 编码器 → 向量量化（离散 token / 行为原语）→ 浅解码器重建 → 三项 VQ 损失联合训练。

代码位于 `F:\WorkBuddy\IMU数据处理\src\imuformer\`，本 notebook 仅做调用与训练编排。

> 运行前请确认你的 Jupyter 内核能正常 `import torch`（若出现 `Numpy is not available`，是内核的 torch/numpy C-API 不匹配，与本项目代码无关）。

## 0. 路径 & 导入

接上「数据层」notebook 的环境设置：把 `src` 加入 Python 路径后导入包。

In [1]:
import sys, os, time, math
from pathlib import Path
import torch
from torch.utils.data import DataLoader

SRC = r"F:\WorkBuddy\IMU数据处理\src"
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import imuformer as im
from imuformer.config   import IMUFormerConfig
from imuformer.dataset  import IMUWindowDataset

print("PyTorch:", torch.__version__)
print("CUDA 可用:", torch.cuda.is_available())

PyTorch: 2.5.1+cu121
CUDA 可用: True


## 1. 配置

所有超参数集中在 `IMUFormerConfig`：2s@256Hz 窗、patch 64/stride 32、共享编码器、K=256 码本、纯 VQ（λ=0）。

In [2]:
cfg = IMUFormerConfig()
print(cfg.summary())

# 纯 VQ 最小验证（默认即是）：λ=0, β=0.25, K=256, 2s 窗
# 如需改超参，例如： cfg = IMUFormerConfig(codebook_size=256, commit_beta=0.25)

════════ IMU-Former 配置摘要 ════════
  采样率:          256 Hz
  窗口:            2.0s = 512 点
  步进:            1.0s = 256 点 (50% 重叠)
  Patch:           P=64 (0.250s), S=32 (0.125s)
  Patch/模态:      16
  模态数:          3 (gravity, accel, gyro)
  总 token 数:     48
  Patch 展平维度:  192 (= 64 × 3)
  模型维度 D:      128
  码本大小 K:      256
  编码器层数:      4
  解码器层数:      2
  承诺损失 β:      0.25
  频域辅助 λ:      0.0 (纯VQ)
  单位缩放:        已设(gyro→rad/s)
  码本更新:        EMA+死码重置
  码本初始化量级:  1.0
══════════════════════════════════════


## 2. 数据集（多段 CSV 合并为一个训练集）

`csv_paths` 传 list 即可合并多段记录；单文件约 268MB 也能直接吃，数据层做了逐文件惰性读取。

In [4]:
import sys, os, glob
from pathlib import Path

# 把“包含 src/ 的目录”加进模块搜索路径（训练 notebook 也是这么干的）
# 你的 import 写法是 `from src.imuformer...`，所以这里加【项目根目录】（即含 src 的那一层）
PROJECT_ROOT = r"F:/WorkBuddy/IMU数据处理"   # ← src 所在的那一层目录，按需改
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.imuformer.dataset import IMUWindowDataset

# ===== 批量提取 =====
DATA_DIR = r"\\hulab.lumiani.ai\Data\WSB\Computational_Ethology\IMU_Data\processed"
CSV_PATHS = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
EXCLUDE = ("timeline", "manifest", "codes", "interp")
CSV_PATHS = [p for p in CSV_PATHS if not any(k in os.path.basename(p).lower() for k in EXCLUDE)]

print(f"自动收集到 {len(CSV_PATHS)} 个训练 csv：")
for p in CSV_PATHS:
    print("  ", p)

# ===== 接训练（确保 cfg 已在前面的 cell 定义）=====
dataset = IMUWindowDataset(csv_paths=CSV_PATHS, config=cfg)
print(f"训练集窗口数: {len(dataset)}")
print(f"≈ {len(dataset) * cfg.window_sec / 3600:.2f} 小时 @ {cfg.sample_rate}Hz")



自动收集到 8 个训练 csv：
   \\hulab.lumiani.ai\Data\WSB\Computational_Ethology\IMU_Data\processed\test1_test4_13.csv
   \\hulab.lumiani.ai\Data\WSB\Computational_Ethology\IMU_Data\processed\test1_test4_ab.csv
   \\hulab.lumiani.ai\Data\WSB\Computational_Ethology\IMU_Data\processed\test1_tool1_13.csv
   \\hulab.lumiani.ai\Data\WSB\Computational_Ethology\IMU_Data\processed\test1_tool1_e8.csv
   \\hulab.lumiani.ai\Data\WSB\Computational_Ethology\IMU_Data\processed\test2_tool1_13.csv
   \\hulab.lumiani.ai\Data\WSB\Computational_Ethology\IMU_Data\processed\test2_tool1_e8.csv
   \\hulab.lumiani.ai\Data\WSB\Computational_Ethology\IMU_Data\processed\test3_tool1_13.csv
   \\hulab.lumiani.ai\Data\WSB\Computational_Ethology\IMU_Data\processed\test3_tool1_ab.csv
训练集窗口数: 12572
≈ 6.98 小时 @ 256Hz


## 3. DataLoader

这就是「数据层」末尾的 `loader`。`batch['x']` 形状 `(B, 512, 9)` 即喂给模型的总输入 `x`；`patches (B,48,192)` 是中间产物——模型内部已含 patcher，你喂 raw window 即可。

In [5]:
BATCH_SIZE = 32
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,     # Windows 下建议 0，避免多进程报错
    drop_last=True,    # 丢弃最后一个不满批的窗口
)

batch = next(iter(loader))
x, mask = batch["x"], batch["mask"]
print("x   :", tuple(x.shape))      # (32, 512, 9)  ← 这就是喂给 model 的 x
print("mask:", tuple(mask.shape))   # (32, 512)

x   : (32, 512, 9)
mask: (32, 512)


F:\WorkBuddy/IMU数据处理\src\imuformer\dataset.py:46: UserWarning: The given buffer is not writable, and PyTorch does not support non-writable tensors. This means you can write to the underlying (supposedly non-writable) buffer using the tensor. You may want to copy the buffer to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\torch\csrc\utils\tensor_new.cpp:1566.)
  return torch.frombuffer(


## 4. 模型 + 优化器 + 设备

`IMUFormer` 统合了 patcher + encoder + quantizer + decoder。默认 CPU；有 GPU 改 `device="cuda"`。

In [6]:
device = torch.device("cpu")        # 有 GPU 改成 torch.device("cuda")
model = im.IMUFormer(cfg).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

n_params = sum(p.numel() for p in model.parameters())
print(f"参数量: {n_params:,}")
print(f"设备:   {device}")

d:\Anaconda3\envs\DEEPLABCUT\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


参数量: 1,274,304
设备:   cpu


## 5. 码本健康监控函数

`perplexity`（困惑度）衡量码本被有效使用的数量，健康值 ≥ ~85（K=256 时）；`usage` 为被使用过的码占比。训练中应逐步上升，过低 = 码本坍缩。

In [7]:
@torch.no_grad()
def evaluate_codebook(model, dataset, batch_size=64, max_windows=2000):
    """遍历数据集抽 token，统计困惑度与码本利用率。"""
    model.eval()
    all_codes = []
    dl = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    n = 0
    for b in dl:
        codes = model.encode_codes(b["x"].to(device))   # (B, 48)
        all_codes.append(codes.reshape(-1))
        n += codes.shape[0]
        if n >= max_windows:
            break
    all_codes = torch.cat(all_codes, dim=0)
    pp  = model.quantizer.perplexity(all_codes)
    use = model.quantizer.codebook_usage(all_codes)
    return pp, use

## 6. 训练循环（核心）

每个 batch：`model(x, mask=mask)` 内部自动 patching→encode→VQ→decode，返回 `total_loss`；`mask` 传入后插值位置在重建损失里被降权（权重 `1-mask*0.9`）。每 epoch 监控码本健康并保存检查点。

In [8]:
EPOCHS   = 50
LOG_EVERY = 50
CKPT_DIR = Path(r"F:\WorkBuddy\IMU数据处理\outputs\imuformer_ckpt")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

model.train()
global_step = 0
best_pp = 0.0

for epoch in range(EPOCHS):
    epoch_loss, n_batches, t0 = 0.0, 0, time.time()

    for batch in loader:
        x    = batch["x"].to(device)      # (B, 512, 9)  raw window
        mask = batch["mask"].to(device)   # (B, 512)    插值掩码

        optimizer.zero_grad()
        out  = model(x, mask=mask)        # 内部自动 patching→encode→VQ→decode
        loss = out["total_loss"]          # = 重建 + 码本 + β·承诺 + λ·频域(=0)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        n_batches  += 1
        global_step += 1

        if global_step % LOG_EVERY == 0:
            print(f"[step {global_step:5d} | ep {epoch+1}/{EPOCHS}] "
                  f"total={loss.item():.4f}  recon={out['recon_loss'].item():.4f}  "
                  f"cb={out['codebook_loss'].item():.4f}  cm={out['commit_loss'].item():.4f}")

    avg = epoch_loss / max(n_batches, 1)
    print(f"══ Epoch {epoch+1}/{EPOCHS}  avg_loss={avg:.4f}  ({time.time()-t0:.1f}s) ═")

    # 每 epoch 监控码本健康
    pp, use = evaluate_codebook(model, dataset)
    print(f"    codebook: perplexity={pp:.1f} (目标≥~85)  usage={use:.3f}")

    # 保存检查点（每 epoch 覆盖 + 按 perplexity 择优）
    torch.save({
        "epoch": epoch + 1,
        "global_step": global_step,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "config": cfg,
        "perplexity": pp, "usage": use,
    }, CKPT_DIR / f"imuformer_ep{epoch+1:03d}.pt")

    if pp > best_pp:
        best_pp = pp
        torch.save({
            "epoch": epoch + 1,
            "global_step": global_step,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "config": cfg,
            "perplexity": pp, "usage": use,
        }, CKPT_DIR / "imuformer_best.pt")
        print(f"    ★ 新最佳 perplexity，已保存 best.pt")

    model.train()   # evaluate_codebook 里切过 eval，这里切回

[step    50 | ep 1/50] total=2.3168  recon=1.3092  cb=4.0300  cm=4.0300
[step   100 | ep 1/50] total=2.3885  recon=1.2975  cb=4.3639  cm=4.3639
[step   150 | ep 1/50] total=2.0480  recon=1.0826  cb=3.8618  cm=3.8618
[step   200 | ep 1/50] total=1.7289  recon=0.8902  cb=3.3548  cm=3.3548
[step   250 | ep 1/50] total=1.7191  recon=0.8676  cb=3.4062  cm=3.4062
[step   300 | ep 1/50] total=1.9933  recon=0.9919  cb=4.0058  cm=4.0058
[step   350 | ep 1/50] total=1.9303  recon=0.9337  cb=3.9864  cm=3.9864
══ Epoch 1/50  avg_loss=2.0590  (34.8s) ═
    codebook: perplexity=146.1 (目标≥~85)  usage=0.914
    ★ 新最佳 perplexity，已保存 best.pt
[step   400 | ep 2/50] total=1.8759  recon=0.8972  cb=3.9147  cm=3.9147
[step   450 | ep 2/50] total=1.6129  recon=0.7803  cb=3.3303  cm=3.3303
[step   500 | ep 2/50] total=1.7957  recon=0.8602  cb=3.7417  cm=3.7417
[step   550 | ep 2/50] total=1.7733  recon=0.8484  cb=3.6994  cm=3.6994
[step   600 | ep 2/50] total=2.0776  recon=0.9954  cb=4.3288  cm=4.3288
[step   

## 7. 训练完成后：抽取全部 token 供下游行为分析

把整段数据量化成 token 序列 `(n_windows, 48)`，供下游 n-gram + GMM / 朴素贝叶斯 行为聚类使用。

In [10]:
@torch.no_grad()
def extract_all_codes(model, dataset, batch_size=64, save_path=None):
    """整段数据量化成 token 序列 (n_windows, 48)，供下游行为聚类。"""
    model.eval()
    dl = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    code_list = []
    for b in dl:
        code_list.append(model.encode_codes(b["x"].to(device)).cpu())
    all_codes = torch.cat(code_list, dim=0)     # (n_windows, 48)
    if save_path:
        torch.save(all_codes, save_path)
        print(f"已保存 token 序列 → {save_path}  形状 {tuple(all_codes.shape)}")
    return all_codes

all_codes = extract_all_codes(
    model, dataset,
    save_path=r"F:\WorkBuddy\IMU数据处理\outputs\imuformer_ckpt\all_codes.pt"
)
print("token 序列形状:", tuple(all_codes.shape))   # (n_windows, 48)

已保存 token 序列 → F:\WorkBuddy\IMU数据处理\outputs\imuformer_ckpt\all_codes.pt  形状 (5175, 48)
token 序列形状: (5175, 48)


## 8. 加载检查点 / 续训

从 `best.pt` 恢复模型与优化器状态；续训时把 Cell 6 的 `range(EPOCHS)` 改成 `range(start_epoch, EPOCHS)`。

In [11]:
ckpt = torch.load(CKPT_DIR / "imuformer_best.pt", map_location=device)
model.load_state_dict(ckpt["model_state"])
optimizer.load_state_dict(ckpt["optimizer_state"])
start_epoch = ckpt["epoch"]
print(f"已加载: epoch={start_epoch}, perplexity={ckpt['perplexity']:.1f}")

已加载: epoch=4, perplexity=211.2


C:\Users\Shubai\AppData\Local\Temp\ipykernel_19232\2851262119.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(CKPT_DIR / "imuformer_best.pt", map_locat

## 9. 跨模态耦合可视化（可解释性）

对应方案里的决策 2「轴间可能存在有意义的解耦」：直接看出哪些行为下 a（非重力）会去看 ω（角速度）、哪些下解耦。行=query 模态，列=key 模态，值越大表示越关注。

In [12]:
import numpy as np

model.eval()
sample = next(iter(loader))["x"][:4].to(device)
coupling = model.analyze_modality_coupling(sample)   # (4, 3, 3)
print("跨模态耦合 [query 行 → key 列]:")
print(np.round(coupling[0].detach().numpy(), 3))

跨模态耦合 [query 行 → key 列]:
[[0.271 0.367 0.362]
 [0.103 0.378 0.519]
 [0.211 0.406 0.383]]


## 调参 / 排错提示

1. **码本坍缩**：`perplexity` 持续很低（如 <50）或 `usage` 很低 = 大量 token 没被用到。当前 `VectorQuantizer` 的 EMA 抗坍缩开关未接进 `IMUFormer` 构造（默认 `ema=False`），真遇到坍缩需小改 `model.py` 传 `ema=True`。
2. **单位量级**：gyro 值域 ~±120 (deg/s)，gravity/accel ~±1，重建损失会被 gyro 主导。可在 `IMUFormerConfig` 设 `channel_scales=(1,1,1,1,1,1,π/180,π/180,π/180)` 把 gyro 转 rad/s（物理单位换算，非统计归一化）。
3. **频域辅助**：`lambda_freq=0`（纯 VQ 最小验证）。想开时设 `lambda_freq=0.2`，结构已留好、无需改代码。
4. **设备**：CPU 跑 268MB 数据较慢，建议先小段 CSV 验证流程通了再上全量；有 GPU 改 `device="cuda"`。